# Telemetry & CES Analysis

Loads a saved `System0Sandbox` run (config, telemetry, trace, summary, agent snapshot) and
visualizes the energy-aware cognition signals: reward / CES-like efficiency over time,
salience & threshold dynamics, survival rate, energy usage, and prospective-forecast accuracy.

Run `python experiments/run_demo.py` first (or point `RUN_DIR` at any existing `results/<config>/<run_id>/` directory).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from metabolic_intelligence_lab.persistence import latest_run, load_snapshot

CONFIG_NAME = "baseline_demo"
run_dir = latest_run(CONFIG_NAME)
if run_dir is None:
    raise FileNotFoundError(
        f"No saved runs found for '{CONFIG_NAME}'. Run `python experiments/run_demo.py` first."
    )
print(f"Loading run: {run_dir}")
snap = load_snapshot(run_dir)
trace = pd.DataFrame(snap.trace)
trace.head()

## Run summary

In [ ]:
summary = snap.summary
pd.Series(summary)

## Reward & CES-efficiency over time

Pulls the per-tool-call `plan:*` rows out of `telemetry.csv` (each carries `total_reward`/`ces_like`
for the System-2 escalation that produced it) and plots their trend across the run.

In [ ]:
telemetry = pd.read_csv(run_dir / "telemetry.csv")
plans = telemetry[telemetry["tag"].astype(str).str.startswith("plan:")].dropna(subset=["total_reward"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(plans["t"], plans["total_reward"], marker="o")
axes[0].set_title("Total reward per escalation")
axes[0].set_xlabel("t")
axes[0].set_ylabel("total_reward")

axes[1].plot(plans["t"], plans["ces_like"], marker="o", color="darkorange")
axes[1].set_title("CES-like efficiency per escalation")
axes[1].set_xlabel("t")
axes[1].set_ylabel("ces_like")
plt.tight_layout()
plt.show()

## Salience vs. arbitration threshold

Shows when top salience crosses the dynamic threshold — the moments System 0 hands control to System 2.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(trace["t"], trace["top_salience"], label="top_salience")
ax.plot(trace["t"], trace["threshold"], label="threshold", linestyle="--")
ax.scatter(trace.loc[trace["escalated"], "t"], trace.loc[trace["escalated"], "top_salience"],
           color="red", zorder=5, label="escalated")
ax.set_xlabel("t")
ax.set_ylabel("value")
ax.set_title("Salience vs. threshold (red = System-2 escalation)")
ax.legend()
plt.tight_layout()
plt.show()

## Energy usage & survival

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(trace["t"], trace["energy_used"], color="steelblue")
axes[0].set_title("Energy spent per tick")
axes[0].set_xlabel("t")
axes[0].set_ylabel("energy_used")

hunger = trace["world"].apply(lambda w: w["hunger"])
fire_lit = trace["world"].apply(lambda w: w["fire"]["lit"])
axes[1].plot(trace["t"], hunger, label="hunger", color="firebrick")
axes[1].fill_between(trace["t"], 0, hunger.max() if hunger.max() else 1,
                     where=fire_lit, color="orange", alpha=0.15, label="fire lit")
axes[1].set_title("Survival pressure: hunger over time (fire-lit shaded)")
axes[1].set_xlabel("t")
axes[1].set_ylabel("hunger")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"survival_rate (from summary): {summary['survival_rate']:.3f}")

## Prospective-forecast accuracy

`ProspectiveLog.entries` records each tick's forecast, what was actually observed next, and a
Jaccard hit/miss score (1.0 = perfect, 0.0 = complete miss). Plotting the rolling score shows
whether the agent's look-ahead is improving as memory reinforces.

In [ ]:
plog = pd.DataFrame(snap.agent_state["plog"]["entries"])
plog = plog.dropna(subset=["score"])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plog["t"], plog["score"], alpha=0.4, label="per-tick score")
ax.plot(plog["t"], plog["score"].rolling(5, min_periods=1).mean(), color="crimson", label="rolling mean (5)")
ax.set_title("Prospective forecast accuracy (Jaccard overlap of forecast vs. observed)")
ax.set_xlabel("t")
ax.set_ylabel("score")
ax.legend()
plt.tight_layout()
plt.show()

print(f"avg score: {plog['score'].mean():.3f}   hit rate: {(plog['score'] > 0).mean():.3f}")

## Goal-frontier composition

Which goals dominate the Pareto frontier (urgency vs. value) across the run, and how often each was actually chosen.

In [ ]:
goal_counts = trace["goal"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
goal_counts.plot(kind="bar", ax=ax, color="slateblue")
ax.set_title("Chosen-goal frequency")
ax.set_xlabel("goal")
ax.set_ylabel("ticks")
plt.tight_layout()
plt.show()